# 18 · **SR vs task horizon** — transfer(짧) → insertion(긺)

논문의 핵심 그림. 같은 모델의 두 task SR 을 나란히 놓고 **격차가 horizon 과 함께 벌어지는지** 본다.
- Intro 주장: "짧은 task 는 ACT 와 대등, 길수록 우리가 앞선다."
- 두 task 모두 같은 프로토콜(150k · 4 seed · 5 rep × 500 ep)이라 그대로 비교 가능.
- 아직 eval 안 된 (모델, task) 는 자동으로 빠진다.

In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASKS_AXIS = [cf.SHORT_SIM, cf.MAIN_SIM]      # 짧 → 긺
TAGS  = cf.FINAL_TAGS                          # 있는 것만 그려짐
SEEDS = cf.MAIN_SEEDS
REPS  = list(range(cf.EVAL_REPEATS))
N_EP  = cf.EVAL_N_EP

print('horizon 축:', TASKS_AXIS)
for tk in TASKS_AXIS:
    done = [t for t in TAGS if cf.sr_over_reps(t, task=tk, seeds=SEEDS, reps=REPS)['mean'] is not None]
    print(f'  {tk:<10} eval 완료: {done or "(없음)"}')

## task 별 SR 표

In [ ]:
for tk in TASKS_AXIS:
    print(f'\n===== {tk} =====')
    cf.sr_table(TAGS, SEEDS, REPS, task=tk, n_episodes=N_EP)

## 그림 — SR vs horizon

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
                     'font.size': 13, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

out = cf.OUTPUT_BASE / 'main_report'
out.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))
drawn = 0
for t in TAGS:
    xs, ys = [], []
    for i, tk in enumerate(TASKS_AXIS):
        agg = cf.sr_over_reps(t, task=tk, seeds=SEEDS, reps=REPS)
        if agg['mean'] is not None:
            xs.append(i)
            ys.append(agg['mean'])
    if not ys:
        continue
    drawn += 1
    ax.plot(xs, ys, '-o', ms=7, color=cf.COLOR.get(t, '#333'), label=cf.FINAL_LABELS.get(t, t))

if not drawn:
    print('아직 결과 없음 — 두 task 를 eval 한 뒤 다시 실행')
else:
    ax.set_xticks(range(len(TASKS_AXIS)))
    ax.set_xticklabels(['transfer\n(short)', 'insertion\n(long horizon)'][:len(TASKS_AXIS)])
    ax.set_ylabel('Success rate (%)')
    ax.set_title('SR vs task horizon', fontweight='bold')
    ax.legend(fontsize=10)
    fig.savefig(out / 'sr_vs_horizon.png')
    fig.savefig(out / 'sr_vs_horizon.pdf')
    plt.show()
    print('저장:', out / 'sr_vs_horizon.png')